<a href="https://colab.research.google.com/github/MilagrosAlonso/TP-IA/blob/main/agente_ventas_mistral_corregido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agente de Análisis de Ventas
### LangChain + Pandas + Mistral Small + RAG (FAISS)
Dataset: `sales_data_normalized.csv`

## 1. Instalación de dependencias

In [1]:
# Instalar todas las dependencias en una sola celda (sin --force-reinstall)
!pip install -q -U langchain langchain-mistralai langchain-experimental pandas tabulate
!pip install -q faiss-cpu langchain-community sentence-transformers scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

## 2. Configuración de credenciales

In [2]:
import os

# Opción A: setear directamente (solo para pruebas locales)
# os.environ["MISTRAL_API_KEY"] = "tu_api_key_aqui"

# Opción B: Colab Secrets (recomendado)
# from google.colab import userdata
# os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

# Opción C: input manual
import getpass
if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Ingresá tu Mistral API Key: ")

print("✅ API Key configurada")

Ingresá tu Mistral API Key: ··········
✅ API Key configurada


## 3. Carga del dataset

In [3]:
import pandas as pd
from google.colab import files

# Subir el archivo CSV
uploaded = files.upload()

# Tomar automáticamente el nombre del archivo subido
nombre_archivo = list(uploaded.keys())[0]
print(f"📂 Archivo subido: {nombre_archivo}")

df = pd.read_csv(nombre_archivo, parse_dates=["orderdate"])

print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"📅 Período: {df['orderdate'].min().date()} → {df['orderdate'].max().date()}")
print(f"\nColumnas disponibles:")
print(df.dtypes)

Saving sales_data_normalized.csv to sales_data_normalized.csv
📂 Archivo subido: sales_data_normalized.csv
✅ Dataset cargado: 2823 filas x 22 columnas
📅 Período: 2003-01-06 → 2005-05-31

Columnas disponibles:
ordernumber                  int64
quantityordered              int64
priceeach                  float64
orderlinenumber              int64
sales                      float64
orderdate           datetime64[us]
status                         str
productline                    str
msrp                         int64
productcode                    str
customername                   str
phone                          str
addressline1                   str
addressline2                   str
city                           str
state                          str
postalcode                     str
country                        str
territory                      str
contactlastname                str
contactfirstname               str
dealsize                       str
dtype: object


In [4]:
# Vista previa
df.head(3)

,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,productline,msrp,productcode,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,Motorcycles,95,S10_1678,...,897 Long Airport Avenue,0,NYC,NY,10022,USA,0,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,Motorcycles,95,S10_1678,...,59 rue de l'Abbaye,0,Reims,0,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,Motorcycles,95,S10_1678,...,27 rue du Colonel Pierre Avia,0,Paris,0,75508,France,EMEA,Da Cunha,Daniel,Medium


## 4. Inicialización del LLM (Mistral Small)

In [5]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    max_retries=3,
)

print(f"✅ LLM listo: {llm.model}")

✅ LLM listo: mistral-small-latest


## 5. Creación del Agente Pandas

In [6]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling",
    prefix="""Eres un analista de ventas experto. Tienes acceso a un dataset de ventas
con 2823 registros. El dataframe se llama `df` y contiene las columnas:
ordernumber, quantityordered, priceeach, orderlinenumber, sales, orderdate,
status, productline, msrp, productcode, customername, phone, addressline1,
addressline2, city, state, postalcode, country, territory, contactlastname,
contactfirstname, dealsize.

Responde siempre en español, con claridad y precisión.
Cuando sea útil, formatea los números con separadores de miles.""",
)

print("✅ Agente creado y listo para consultas")

/tmp/ipykernel_2255/3975958248.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent


✅ Agente creado y listo para consultas


## 6. Función auxiliar para consultas

In [7]:
def preguntar(consulta: str):
    """Envía una consulta al agente y muestra la respuesta."""
    print(f"\n{'='*60}")
    print(f"🧑 Pregunta: {consulta}")
    print(f"{'='*60}")
    respuesta = agent.invoke(consulta)
    print(f"\n🤖 Respuesta final:")
    print(respuesta["output"])
    print(f"{'='*60}\n")
    return respuesta["output"]

## 7. Ejemplos de consultas al agente

Podés modificarlas o agregar las tuyas.

In [8]:
# Consulta 1: Resumen general
preguntar("¿Cuáles son las ventas totales y el número de órdenes en el dataset?")


🧑 Pregunta: ¿Cuáles son las ventas totales y el número de órdenes en el dataset?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'ventas_totales = df[\'sales\'].sum()\nnumero_ordenes = df[\'ordernumber\'].nunique()\n\nventas_totales_formateado = f"{ventas_totales:,.2f}"\nnumero_ordenes_formateado = f"{numero_ordenes:,}"\n\nventas_totales_formateado, numero_ordenes_formateado'}`


('10,032,628.85', '307')Las ventas totales en el dataset son **10.032.628,85** y el número total de órdenes es **307**.

> Finished chain.

🤖 Respuesta final:
Las ventas totales en el dataset son **10.032.628,85** y el número total de órdenes es **307**.



'Las ventas totales en el dataset son **10.032.628,85** y el número total de órdenes es **307**.'

In [9]:
# Consulta 2: Ranking de líneas de productos
preguntar("¿Cuál es la línea de productos más vendida por total de ventas? Mostrá el ranking completo.")


🧑 Pregunta: ¿Cuál es la línea de productos más vendida por total de ventas? Mostrá el ranking completo.


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ranking_productline = df.groupby('productline')['sales'].sum().sort_values(ascending=False)\nranking_productline"}`


productline
Classic Cars        3919615.66
Vintage Cars        1903150.84
Motorcycles         1166388.34
Trucks and Buses    1127789.84
Planes               975003.57
Ships                714437.13
Trains               226243.47
Name: sales, dtype: float64El ranking completo de las líneas de productos más vendidas por **total de ventas** (en orden descendente) es el siguiente:

| **Posición** | **Línea de Producto** | **Total de Ventas (USD)** |
|--------------|-----------------------|---------------------------|
| 1            | Classic Cars          | 3.919.615,66              |
| 2            | Vintage Cars          | 1.903.150,84              |
| 3            | Motorcycles      

'El ranking completo de las líneas de productos más vendidas por **total de ventas** (en orden descendente) es el siguiente:\n\n| **Posición** | **Línea de Producto** | **Total de Ventas (USD)** |\n|--------------|-----------------------|---------------------------|\n| 1            | Classic Cars          | 3.919.615,66              |\n| 2            | Vintage Cars          | 1.903.150,84              |\n| 3            | Motorcycles           | 1.166.388,34              |\n| 4            | Trucks and Buses      | 1.127.789,84              |\n| 5            | Planes                |   975.003,57              |\n| 6            | Ships                 |   714.437,13              |\n| 7            | Trains                |   226.243,47              |\n\n**Conclusión:** La línea de productos más vendida es **Classic Cars**, con un total de ventas de **3.919.615,66 USD**.'

In [10]:
# Consulta 3: Top países
preguntar("¿Cuáles son los 5 países con mayor volumen de ventas totales?")


🧑 Pregunta: ¿Cuáles son los 5 países con mayor volumen de ventas totales?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "ventas_por_pais = df.groupby('country')['sales'].sum().reset_index()\nventas_por_pais = ventas_por_pais.sort_values(by='sales', ascending=False)\nventas_por_pais_top5 = ventas_por_pais.head(5)\nventas_por_pais_top5"}`


      country       sales
18        USA  3627982.83
14      Spain  1215686.92
6      France  1110916.52
0   Australia   630623.10
17         UK   478880.46Los **5 países con mayor volumen de ventas totales** son:

1. **USA**: 3.627.982,83
2. **Spain**: 1.215.686,92
3. **France**: 1.110.916,52
4. **Australia**: 630.623,10
5. **UK**: 478.880,46

> Finished chain.

🤖 Respuesta final:
Los **5 países con mayor volumen de ventas totales** son:

1. **USA**: 3.627.982,83
2. **Spain**: 1.215.686,92
3. **France**: 1.110.916,52
4. **Australia**: 630.623,10
5. **UK**: 478.880,46



'Los **5 países con mayor volumen de ventas totales** son:\n\n1. **USA**: 3.627.982,83\n2. **Spain**: 1.215.686,92\n3. **France**: 1.110.916,52\n4. **Australia**: 630.623,10\n5. **UK**: 478.880,46'

In [11]:
# Consulta 4: Tendencia temporal
preguntar("¿Cómo evolucionaron las ventas mes a mes? Mostrá el total por mes.")


🧑 Pregunta: ¿Cómo evolucionaron las ventas mes a mes? Mostrá el total por mes.


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Asegurarse de que 'orderdate' sea de tipo datetime\ndf['orderdate'] = pd.to_datetime(df['orderdate'])\n\n# Extraer el año y mes para agrupar\nventas_mensuales = df.groupby(\n    df['orderdate'].dt.to_period('M')\n)['sales'].sum().reset_index()\n\n# Convertir el período a un formato más legible\nventas_mensuales['orderdate'] = ventas_mensuales['orderdate'].dt.to_timestamp()\n\n# Ordenar por fecha\nventas_mensuales = ventas_mensuales.sort_values('orderdate')\n\n# Mostrar el resultado\nventas_mensuales"}`


    orderdate       sales
0  2003-01-01   129753.60
1  2003-02-01   140836.19
2  2003-03-01   174504.90
3  2003-04-01   201609.55
4  2003-05-01   192673.11
5  2003-06-01   168082.56
6  2003-07-01   187731.88
7  2003-08-01   197809.30
8  2003-09-01   263973.36
9  2003-10-01   568290.97
10 2003-11-01

'### Evolución de las ventas mes a mes\n\nA continuación, se presenta la evolución del **total de ventas (sales)** mes a mes, en formato de tabla y con los valores formateados para mejor legibilidad:\n\n| **Mes/Año**   | **Ventas Totales (USD)** |\n|---------------|--------------------------|\n| Enero 2003    | 129.754 |\n| Febrero 2003  | 140.836 |\n| Marzo 2003    | 174.505 |\n| Abril 2003    | 201.610 |\n| Mayo 2003     | 192.673 |\n| Junio 2003    | 168.083 |\n| Julio 2003    | 187.732 |\n| Agosto 2003   | 197.809 |\n| Septiembre 2003 | 263.973 |\n| Octubre 2003  | 568.291 |\n| Noviembre 2003 | 1.029.838 |\n| Diciembre 2003 | 261.876 |\n| Enero 2004    | 316.577 |\n| Febrero 2004  | 311.420 |\n| Marzo 2004    | 205.734 |\n| Abril 2004    | 206.148 |\n| Mayo 2004     | 273.438 |\n| Junio 2004    | 286.674 |\n| Julio 2004    | 327.144 |\n| Agosto 2004   | 461.501 |\n| Septiembre 2004 | 320.751 |\n| Octubre 2004  | 552.924 |\n| Noviembre 2004 | 1.089.048 |\n| Diciembre 2004 | 372.803 

In [12]:
# Consulta 5: Top clientes
preguntar("¿Cuáles son los 10 clientes que más compraron en términos de ventas totales?")


🧑 Pregunta: ¿Cuáles son los 10 clientes que más compraron en términos de ventas totales?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "top_customers = df.groupby('customername')['sales'].sum().nlargest(10)\ntop_customers"}`


customername
Euro Shopping Channel           912294.11
Mini Gifts Distributors Ltd.    654858.06
Australian Collectors, Co.      200995.41
Muscle Machine Inc              197736.94
La Rochelle Gifts               180124.90
Dragon Souveniers, Ltd.         172989.68
Land of Toys Inc.               164069.44
The Sharp Gifts Warehouse       160010.27
AV Stores, Co.                  157807.81
Anna's Decorations, Ltd         153996.13
Name: sales, dtype: float64Los **10 clientes que más compraron en términos de ventas totales** son los siguientes:

| **Cliente**                     | **Ventas Totales (USD)** |
|----------------------------------|--------------------------|
| Euro Shopping Channel            | 912.294,11          

"Los **10 clientes que más compraron en términos de ventas totales** son los siguientes:\n\n| **Cliente**                     | **Ventas Totales (USD)** |\n|----------------------------------|--------------------------|\n| Euro Shopping Channel            | 912.294,11               |\n| Mini Gifts Distributors Ltd.     | 654.858,06               |\n| Australian Collectors, Co.       | 200.995,41               |\n| Muscle Machine Inc               | 197.736,94               |\n| La Rochelle Gifts                | 180.124,90               |\n| Dragon Souveniers, Ltd.          | 172.989,68               |\n| Land of Toys Inc.                | 164.069,44               |\n| The Sharp Gifts Warehouse        | 160.010,27               |\n| AV Stores, Co.                   | 157.807,81               |\n| Anna's Decorations, Ltd          | 153.996,13               |\n\nEstos resultados reflejan el volumen de ventas acumulado por cada cliente en el dataset. Si necesitas un análisis más detallado

In [13]:
# Consulta 6: Estado de órdenes
preguntar("¿Cuántas órdenes hay en cada estado (Shipped, Cancelled, etc.) y qué porcentaje representan?")


🧑 Pregunta: ¿Cuántas órdenes hay en cada estado (Shipped, Cancelled, etc.) y qué porcentaje representan?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Contar el número de órdenes por estado\nstatus_counts = df['status'].value_counts()\n\n# Calcular el porcentaje que representa cada estado\ntotal_orders = len(df)\nstatus_percentages = (status_counts / total_orders) * 100\n\n# Crear un DataFrame para mostrar los resultados\nstatus_summary = pd.DataFrame({\n    'Número de Órdenes': status_counts,\n    'Porcentaje (%)': status_percentages.round(2)\n})\n\nstatus_summary"}`


            Número de Órdenes  Porcentaje (%)
status                                       
Shipped                  2617           92.70
Cancelled                  60            2.13
Resolved                   47            1.66
On Hold                    44            1.56
In Process                 41            1.45
Disputed                   14      

'### Resumen de órdenes por estado\n\n| **Estado**    | **Número de Órdenes** | **Porcentaje (%)** |\n|---------------|-----------------------|--------------------|\n| Shipped       | 2.617                 | 92,70%             |\n| Cancelled     | 60                    | 2,13%              |\n| Resolved      | 47                    | 1,66%              |\n| On Hold       | 44                    | 1,56%              |\n| In Process    | 41                    | 1,45%              |\n| Disputed      | 14                    | 0,50%              |\n\n**Total de órdenes analizadas:** 2.823\n\n---\n**Conclusión:**\nEl **92,70%** de las órdenes están en estado **"Shipped"**, lo que indica que la mayoría de las transacciones se completaron con éxito. Los demás estados (Cancelled, Resolved, On Hold, In Process y Disputed) representan menos del **8%** del total, siendo **"Cancelled"** el segundo estado más frecuente con un **2,13%**.'

## 8. Chat interactivo (Agente Pandas)

Escribí tus propias preguntas. Ingresá `salir` para terminar.

In [ ]:
print("💬 Modo interactivo — escribí tu pregunta (o 'salir' para terminar)\n")

while True:
    consulta = input("Vos: ").strip()
    if not consulta:
        continue
    if consulta.lower() in ["salir", "exit", "quit"]:
        print("👋 ¡Hasta luego!")
        break
    preguntar(consulta)

💬 Modo interactivo — escribí tu pregunta (o 'salir' para terminar)

Vos: cual es el producto mas vendido

🧑 Pregunta: cual es el producto mas vendido


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "producto_mas_vendido = df.groupby('productcode')['quantityordered'].sum().idxmax()\nproducto_mas_vendido_info = df[df['productcode'] == producto_mas_vendido].iloc[0]\nproducto_mas_vendido_info[['productcode', 'productline', 'msrp']]"}`


productcode        S18_3232
productline    Classic Cars
msrp                    169
Name: 1013, dtype: objectEl producto más vendido en términos de **cantidad ordenada** es:

- **Código del producto:** `S18_3232`
- **Línea de producto:** `Classic Cars`
- **Precio de venta sugerido (MSRP):** **169**

Si necesitas más detalles, como el nombre del cliente que más lo compró o el territorio con mayores ventas de este producto, puedo proporcionártelo.

> Finished chain.

🤖 Respuesta final:
El producto más vendido en términos d

---
## 9. Módulo RAG con FAISS

> **Importante:** esta sección usa el mismo `df` ya cargado arriba.
> No reinstales paquetes ni reinicies el runtime entre medias.

### 9.1 Convertir cada fila del CSV a documento de texto

In [ ]:
from langchain_core.documents import Document

documentos = []
for _, fila in df.iterrows():
    texto = (
        f"Orden {fila['ordernumber']}: cliente {fila['customername']} "
        f"de {fila['city']}, {fila['country']}. "
        f"Producto: {fila['productline']} (código {fila['productcode']}). "
        f"Cantidad: {fila['quantityordered']}, precio unitario: ${fila['priceeach']}, "
        f"venta total: ${fila['sales']}. "
        f"Estado: {fila['status']}, tamaño del deal: {fila['dealsize']}, "
        f"territorio: {fila['territory']}. "
        f"Fecha: {fila['orderdate']}."
    )
    documentos.append(
        Document(page_content=texto, metadata={"ordernumber": int(fila["ordernumber"])})
    )

print(f"✅ {len(documentos)} documentos creados (1 por fila del CSV)")
print(f"\nEjemplo de documento:")
print(documentos[0].page_content)

### 9.2 Crear embeddings y vector store FAISS

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("⏳ Generando embeddings (puede tardar 1-2 minutos)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"}
)

vector_store = FAISS.from_documents(documentos, embeddings)

print(f"✅ Vector store creado con {vector_store.index.ntotal} embeddings")

### 9.3 Crear el retriever

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("✅ Retriever listo")

# Prueba rápida
docs = retriever.invoke("ventas de motos en USA")
print(f"\n🔍 Prueba — documentos recuperados para 'ventas de motos en USA':")
for i, d in enumerate(docs):
    print(f"  [{i+1}] {d.page_content[:120]}...")

### 9.4 Crear la cadena RAG

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""Eres un analista de ventas experto. Usá el siguiente contexto extraído del dataset de ventas para responder la pregunta.
Si la información no está en el contexto, decilo claramente.
Respondé siempre en español, con claridad y precisión.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template}
)

print("✅ Cadena RAG lista")

### 9.5 Función para consultar el RAG

In [ ]:
def preguntar_rag(consulta: str):
    print(f"\n{'='*60}")
    print(f"🧑 Pregunta (RAG): {consulta}")
    print(f"{'='*60}")

    resultado = rag_chain.invoke({"query": consulta})

    print(f"\n🤖 Respuesta:")
    print(resultado["result"])

    print(f"\n📄 Documentos recuperados ({len(resultado['source_documents'])}):")
    for i, doc in enumerate(resultado["source_documents"]):
        print(f"  [{i+1}] {doc.page_content[:100]}...")

    print(f"{'='*60}\n")
    return resultado["result"]

### 9.6 Ejemplos de consultas RAG

In [ ]:
preguntar_rag("¿Qué órdenes de motocicletas hubo en USA?")

In [ ]:
preguntar_rag("¿Cuáles son las órdenes canceladas y de qué clientes son?")

In [ ]:
preguntar_rag("Dame ejemplos de ventas grandes (Large) en Francia")

### 9.7 Chat interactivo RAG

In [ ]:
print("💬 Modo RAG interactivo — escribí tu pregunta (o 'salir' para terminar)\n")

while True:
    consulta = input("Vos: ").strip()
    if not consulta:
        continue
    if consulta.lower() in ["salir", "exit", "quit"]:
        print("👋 ¡Hasta luego!")
        break
    preguntar_rag(consulta)